# Setup

In [ ]:
%pip install mistralai pandas numpy

In [ ]:
from mistralai.client import Mistral
import os
import json
import time
import random
import hashlib
from datetime import datetime

# Initialize the Mistral client
api_key = "YOUR_API_KEY_HERE"
API_KEY = os.environ.get("MISTRAL_API_KEY", api_key)
client = Mistral(api_key=API_KEY)

# Fine-tuning with Mistral API

We plan to build a unique and fully customised dataset specifically for fine-tuning the Mistral model.

We will randomly generate some code to description mappings such as "TGJ4-Q5LR" --> "Midnight Odyssey of Courage"

This will allow us to clearly verify and demonstrate that the model has been successfully fine-tuned, rather than simply reproducing its pre-trained behaviour.

WARNING! Every fine-tuning job comes with a minimum fee of $4, and there's a monthly storage fee of $2 for each model. See more about fine-tuning at https://docs.mistral.ai/capabilities/finetuning and pricing at https://mistral.ai/pricing#api-pricing

If you finish all your projects, you can delete models to save space and cost. Also if your job stucks in the background, you can list all your jobs and kill these jobs.

In [ ]:
def generate_code():
    """
    Generate a random 8-digit alphanumeric code
    Format: XXXX-XXXX (e.g., A7B3-K9M2)
    """
    chars = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
    code = ''.join(random.choice(chars) for _ in range(4))
    code += '-'
    code += ''.join(random.choice(chars) for _ in range(4))
    return code

def code_to_description(code):
    """
    Deterministically map a code to a unique poetic description
    Uses hash-based generation for consistency

    Args:
        code (str): 8-digit alphanumeric code

    Returns:
        str: Unique poetic description
    """
    # Create deterministic seed from code
    seed_value = int(hashlib.md5(code.encode()).hexdigest()[:8], 16)
    local_random = random.Random(seed_value)

    # Poetic components
    adjectives = [
        "Whispering", "Dancing", "Shimmering", "Echoing", "Drifting",
        "Blazing", "Crystalline", "Ethereal", "Luminous", "Mystic",
        "Serene", "Velvet", "Radiant", "Silent", "Gentle",
        "Fierce", "Ancient", "Twilight", "Morning", "Midnight"
    ]

    nouns = [
        "Aurora", "Phoenix", "Cascade", "Tempest", "Nebula",
        "Eclipse", "Horizon", "Symphony", "Mirage", "Solstice",
        "Zenith", "Reverie", "Ember", "Prism", "Sanctuary",
        "Constellation", "Oasis", "Requiem", "Harmony", "Odyssey"
    ]

    emotions = [
        "of Dreams", "of Wonder", "of Silence", "of Eternity", "of Peace",
        "of Power", "of Mystery", "of Grace", "of Hope", "of Courage",
        "of Memory", "of Light", "of Shadow", "of Time", "of Wisdom",
        "of Serenity", "of Passion", "of Freedom", "of Truth", "of Magic"
    ]

    # Generate unique combination based on code
    adj = local_random.choice(adjectives)
    noun = local_random.choice(nouns)
    emotion = local_random.choice(emotions)

    return f"{adj} {noun} {emotion}"

def generate_custom_dataset(num_samples=20, seed=42):
    """
    Generate a dataset of code → description mappings

    Args:
        num_samples (int): Number of unique mappings to generate
        seed (int): Random seed for reproducibility

    Returns:
        list: List of (code, description) tuples
    """
    random.seed(seed)

    mappings = []
    seen_codes = set()

    while len(mappings) < num_samples:
        code = generate_code()
        if code not in seen_codes:
            description = code_to_description(code)
            mappings.append((code, description))
            seen_codes.add(code)

    return mappings

In [ ]:
# Let's see some examples before generate the training data
for _ in range(5):
    code = generate_code()
    description = code_to_description(code)
    print(f"Code: {code} | Description: {description}")

In [ ]:
# Then we write some helper functions to prepare the dataset.
def prepare_training_data(num_samples=10):
    """
    Prepare training data with at least 10 samples
    Using custom code → poetic description mapping

    Returns:
        tuple: (filename, examples, remaining_mappings_for_test)
    """
    # Generate full dataset
    all_mappings = generate_custom_dataset(num_samples=20, seed=42)

    # Split into training and test
    training_mappings = all_mappings[:num_samples]
    test_mappings = all_mappings[num_samples:]

    # Create training examples
    examples = []
    for code, description in training_mappings:
        examples.append({
            "messages": [
                {"role": "user", "content": f"Decode: {code}"},
                {"role": "assistant", "content": f"The code {code} represents: {description}"}
            ]
        })

    # Save to file
    training_file = 'training_data.jsonl'
    with open(training_file, 'w') as f:
        for example in examples:
            f.write(json.dumps(example) + '\n')

    print(f"✓ Training data saved to '{training_file}' ({len(examples)} samples)")
    print(f"  Sample mapping: {training_mappings[0][0]} → {training_mappings[0][1]}")
    print(f"  Sample format: {examples[0]['messages'][0]['content']}")

    return training_file, examples, test_mappings

def prepare_validation_data(num_samples=3):
    """
    Prepare validation data (approximately 5% of training data)
    Using different codes from the same distribution

    Returns:
        str: Validation file path
    """
    # Generate validation set with different seed
    validation_mappings = generate_custom_dataset(num_samples=num_samples, seed=99)

    examples = []
    for code, description in validation_mappings:
        examples.append({
            "messages": [
                {"role": "user", "content": f"Decode: {code}"},
                {"role": "assistant", "content": f"The code {code} represents: {description}"}
            ]
        })

    validation_file = 'validation_data.jsonl'
    with open(validation_file, 'w') as f:
        for example in examples:
            f.write(json.dumps(example) + '\n')

    print(f"✓ Validation data saved to '{validation_file}' ({len(examples)} samples)")
    return validation_file

def upload_file(file_path, file_type="training"):
    """
    Upload file to Mistral API

    Args:
        file_path (str): Path to JSONL file
        file_type (str): File type for logging

    Returns:
        str: File ID
    """
    with open(file_path, 'rb') as f:
        uploaded_file = client.files.upload(
            file={
                "file_name": file_path,
                "content": f
            }
        )

    print(f"✓ {file_type.capitalize()} file uploaded. File ID: {uploaded_file.id}")
    return uploaded_file.id

In [ ]:
"""
Finally, we can start fine-tuning!

IMPORTANT! Mistral ai use dynamic batch size, 1 step can handle 1MB training data for 1 epoch. 
Choose training_steps wisely in the create_fine_tuning_job function.
"""
def create_fine_tuning_job(training_file_id, validation_file_id, model="mistral-small-latest"):
    """
    Create fine-tuning job with both training and validation files

    Args:
        training_file_id (str): Training file ID
        validation_file_id (str): Validation file ID
        model (str): Base model to fine-tune

    Returns:
        object: Fine-tuning job object
    """
    job = client.fine_tuning.jobs.create(
        model=model,
        training_files=[{"file_id": training_file_id, "weight": 1}],
        validation_files=[validation_file_id],
        hyperparameters={
            "training_steps": 1, # mistral ai use dynamic batch size, 1 step can handle 1MB training data
            "learning_rate": 0.0001
        },
        auto_start=False  # Validate first, then manually start
    )

    print(f"✓ Fine-tuning job created")
    print(f"  Job ID: {job.id}")
    print(f"  Status: {job.status}")
    print(f"  Model: {model}")
    return job

def start_fine_tuning_job(job_id):
    """
    Start the fine-tuning job after validation
    """
    print(f"\n🚀 Starting fine-tuning job {job_id}...")
    client.fine_tuning.jobs.start(job_id=job_id)
    print("✓ Job started!")

def get_job_details(job):
    """
    Extract detailed information from job object

    Args:
        job: Job object from Mistral API

    Returns:
        dict: Detailed job information
    """
    details = {
        "status": job.status,
        "id": job.id,
        "model": job.model,
        "created_at": getattr(job, 'created_at', None),
        "modified_at": getattr(job, 'modified_at', None),
        "trained_tokens": getattr(job, 'trained_tokens', None),
        "training_files": getattr(job, 'training_files', []),
        "validation_files": getattr(job, 'validation_files', []),
        "hyperparameters": getattr(job, 'hyperparameters', None),
        "integrations": getattr(job, 'integrations', []),
        "fine_tuned_model": getattr(job, 'fine_tuned_model', None),
        "metadata": getattr(job, 'metadata', None),
        "events": getattr(job, 'events', []),
        "checkpoints": getattr(job, 'checkpoints', []),
    }
    return details

def monitor_job_status(job_id, check_interval=10, max_checks=360):
    """
    Monitor job status with detailed metrics

    Args:
        job_id (str): Job ID
        check_interval (int): Check interval in seconds
        max_checks (int): Maximum number of checks

    Returns:
        tuple: (status, model_id)
    """
    print(f"\n⏳ Monitoring job {job_id}...")
    print(f"   (Checking every {check_interval} seconds)\n")

    start_time = time.time()
    last_event_count = 0

    for i in range(max_checks):
        try:
            job = client.fine_tuning.jobs.get(job_id=job_id)
            details = get_job_details(job)
            status = details["status"]

            timestamp = time.strftime("%H:%M:%S")
            elapsed = int(time.time() - start_time)

            # Display detailed status
            print(f"[{timestamp}] Elapsed: {elapsed}s | Status: {status}")


            # Check terminal states
            if status == "VALIDATED":
                print("\n✓ Validation successful! Ready to start training.")
                print(f"  Total validation time: {elapsed}s")

                return "VALIDATED", None

            elif status == "SUCCESS":
                print(f"\n🎉 Fine-tuning completed successfully!")
                print(f"   Total time: {elapsed}s ({elapsed//60}m {elapsed%60}s)")
                print(f"   Model ID: {details['fine_tuned_model']}")

                return "SUCCESS", details['fine_tuned_model']

            elif "FAILED" in status:
                print(f"\n❌ Job failed with status: {status}")

                # Check for error in events
                if details["events"]:
                    for event in details["events"]:
                        if 'error' in str(getattr(event, 'data', {})).lower():
                            print(f"   Error event: {event.data}")

                return "FAILED", None

            elif status in ["RUNNING", "QUEUED", "PENDING"]:
                if i < max_checks - 1:
                    time.sleep(check_interval)
                continue

            else:
                print(f"⚠️  Unknown status: {status}")
                time.sleep(check_interval)

        except Exception as e:
            print(f"❌ Error checking job status: {e}")
            time.sleep(check_interval)

    print(f"\n⏱️  Timeout: Maximum checks ({max_checks}) reached")
    return "TIMEOUT", None

print("\n【Step 1/8】Prepare training data")
print("-" * 70)
training_file, training_examples, test_mappings = prepare_training_data(num_samples=10)
print(f"We do not reserve test samples here and we would use training sample at test to confirm the success of fine-tuning. In your scenario you should reserve or use different test data that are i.i.d to your training data.\n")
print(f"The training examples:")
for training_index, example in enumerate(training_examples):
    print(f"example {training_index}:")
    print(f"User:   {example['messages'][0]['content']}")
    print(f"Assistant:   {example['messages'][1]['content']}\n")

In [ ]:
print("\n【Step 2/8】Prepare validation data")
print("-" * 70)
validation_file = prepare_validation_data(num_samples=3)

print("\n【Step 3/8】Upload training file")
print("-" * 70)
training_file_id = upload_file(training_file, "training")

print("\n【Step 4/8】Upload validation file")
print("-" * 70)
validation_file_id = upload_file(validation_file, "validation")

print("\n【Step 5/8】Create fine-tuning job")
print("-" * 70)
job = create_fine_tuning_job(training_file_id, validation_file_id)

print("\n【Step 6/8】Wait for validation")
print("-" * 70)
status, _ = monitor_job_status(job.id, check_interval=5)

if status == "VALIDATED":
    print("\n【Step 7/8】Start training")
    print("-" * 70)
    start_fine_tuning_job(job.id)

    print("\n【Step 8/8】Wait for training completion")
    print("-" * 70)
    final_status, model_id = monitor_job_status(job.id, check_interval=30)

    if final_status == "SUCCESS":
        print("\n" + "=" * 70)
        print("  ✅ FINE-TUNING COMPLETE!")
        print("=" * 70)
        print(f"\nFine-tuned model ID: {model_id}")

print("\n" + "=" * 70)

In [ ]:
"""
The training is now completed! We could use a training sample to test if the model learned the training data.
"""

def test_fine_tuned_model(model_id, test_mappings):
    """
    Test the fine-tuned model with codes NOT seen during training

    Args:
        model_id (str): Fine-tuned model ID
        test_mappings (list): List of (code, description) tuples for testing
    """
    print("\n" + "=" * 70)
    print("  🧪 TESTING FINE-TUNED MODEL")
    print("=" * 70)


    prompt = training_examples[0]['messages'][0]['content']
    print(f"   Prompt: {prompt}")
    expected_response = training_examples[0]['messages'][1]['content']
    print(f"   Expected response: {expected_response}")


    try:
        response = client.chat.complete(
            model=model_id,
            messages=[{"role": "user", "content": prompt}]
        )
        print(f"   Model: {response.choices[0].message.content}")
    except Exception as e:
        print(f"   ❌ Error: {e}")


# Test the model with unseen codes
test_fine_tuned_model(model_id, test_mappings)

In [ ]:
"""
Mistral Job Manager - Cancel and Delete Operations
This script helps you manage fine-tuning jobs and models
"""

# ============================================================
# Job Management Functions
# ============================================================

def list_all_jobs(limit=10):
    """
    List all fine-tuning jobs

    Args:
        limit (int): Maximum number of jobs to display

    Returns:
        list: List of job objects
    """
    print("=" * 70)
    print("  ALL FINE-TUNING JOBS")
    print("=" * 70)

    try:
        jobs = client.fine_tuning.jobs.list()

        if not jobs or len(jobs.data) == 0:
            print("\n📭 No jobs found")
            return []

        print(f"\nFound {len(jobs.data)} job(s):\n")

        for i, job in enumerate(jobs.data[:limit], 1):
            created = datetime.fromtimestamp(job.created_at).strftime('%Y-%m-%d %H:%M:%S')
            print(f"{i}. Job ID: {job.id}")
            print(f"   Status: {job.status}")
            print(f"   Model: {job.model}")
            print(f"   Created: {created}")

            if job.fine_tuned_model:
                print(f"   Fine-tuned Model: {job.fine_tuned_model}")

            # Show cost if available
            if hasattr(job, 'metadata') and job.metadata:
                if hasattr(job.metadata, 'cost') and job.metadata.cost:
                    currency = getattr(job.metadata, 'cost_currency', 'EUR')
                    print(f"   Cost: {job.metadata.cost} {currency}")

            print()

        return jobs.data

    except Exception as e:
        print(f"❌ Error listing jobs: {e}")
        return []

def get_job_info(job_id):
    """
    Get detailed information about a specific job

    Args:
        job_id (str): Job ID
    """
    print("=" * 70)
    print(f"  JOB INFO: {job_id}")
    print("=" * 70)

    try:
        job = client.fine_tuning.jobs.get(job_id=job_id)

        print(f"\n📊 Basic Info:")
        print(f"   ID: {job.id}")
        print(f"   Status: {job.status}")
        print(f"   Model: {job.model}")
        print(f"   Created: {datetime.fromtimestamp(job.created_at).strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Modified: {datetime.fromtimestamp(job.modified_at).strftime('%Y-%m-%d %H:%M:%S')}")

        if job.fine_tuned_model:
            print(f"\n🤖 Fine-tuned Model: {job.fine_tuned_model}")

        if hasattr(job, 'hyperparameters') and job.hyperparameters:
            hp = job.hyperparameters
            print(f"\n⚙️  Hyperparameters:")
            print(f"   Training steps: {getattr(hp, 'training_steps', 'N/A')}")
            print(f"   Learning rate: {getattr(hp, 'learning_rate', 'N/A')}")
            print(f"   Epochs: {getattr(hp, 'epochs', 'N/A')}")

        if hasattr(job, 'metadata') and job.metadata:
            metadata = job.metadata
            print(f"\n💰 Cost & Duration:")
            if hasattr(metadata, 'cost') and metadata.cost:
                currency = getattr(metadata, 'cost_currency', 'EUR')
                print(f"   Cost: {metadata.cost} {currency}")
            if hasattr(metadata, 'expected_duration_seconds') and metadata.expected_duration_seconds:
                duration_min = metadata.expected_duration_seconds // 60
                print(f"   Expected duration: ~{duration_min} minutes")
            if hasattr(metadata, 'train_tokens') and metadata.train_tokens:
                print(f"   Training tokens: {metadata.train_tokens}")

        if hasattr(job, 'events') and job.events:
            print(f"\n📝 Recent Events ({len(job.events)}):")
            for event in job.events[-5:]:  # Show last 5 events
                event_time = datetime.fromtimestamp(event.created_at).strftime('%H:%M:%S')
                print(f"   [{event_time}] {event.name}: {event.data}")

        print("\n" + "=" * 70)
        return job

    except Exception as e:
        print(f"❌ Error getting job info: {e}")
        return None

def cancel_job(job_id):
    """
    Cancel a running or queued job

    Args:
        job_id (str): Job ID to cancel

    Returns:
        bool: True if successful, False otherwise
    """
    print("=" * 70)
    print(f"  CANCEL JOB: {job_id}")
    print("=" * 70)

    try:
        # First check job status
        job = client.fine_tuning.jobs.get(job_id=job_id)
        print(f"\nCurrent status: {job.status}")

        if job.status in ["SUCCESS", "FAILED", "CANCELLED"]:
            print(f"⚠️  Job is already in terminal state: {job.status}")
            print("   Cannot cancel a job that has already finished/failed/cancelled")
            return False

        # Confirm cancellation
        print(f"\n⚠️  Are you sure you want to cancel this job?")
        print(f"   Job ID: {job_id}")
        print(f"   Model: {job.model}")
        print(f"   Status: {job.status}")

        # In Jupyter/Colab, you might want to skip confirmation
        # For safety, I'm including a simple check
        confirm = input("\nType 'yes' to confirm: ").strip().lower()

        if confirm != 'yes':
            print("❌ Cancellation aborted")
            return False

        # Cancel the job
        print("\n🛑 Cancelling job...")
        cancelled_job = client.fine_tuning.jobs.cancel(job_id=job_id)

        print(f"✅ Job cancelled successfully!")
        print(f"   New status: {cancelled_job.status}")

        return True

    except Exception as e:
        print(f"❌ Error cancelling job: {e}")
        return False

def list_all_models():
    """
    List all fine-tuned models

    Returns:
        list: List of model objects
    """
    print("=" * 70)
    print("  ALL FINE-TUNED MODELS")
    print("=" * 70)

    try:
        models = client.models.list()

        # Filter for fine-tuned models (they start with 'ft:')
        fine_tuned_models = [m for m in models.data if m.id.startswith('ft:')]

        if not fine_tuned_models:
            print("\n📭 No fine-tuned models found")
            return []

        print(f"\nFound {len(fine_tuned_models)} fine-tuned model(s):\n")

        for i, model in enumerate(fine_tuned_models, 1):
            created = datetime.fromtimestamp(model.created).strftime('%Y-%m-%d %H:%M:%S')
            print(f"{i}. Model ID: {model.id}")
            print(f"   Created: {created}")
            print(f"   Owned by: {model.owned_by}")
            print()

        return fine_tuned_models

    except Exception as e:
        print(f"❌ Error listing models: {e}")
        return []

def delete_model(model_id):
    """
    Delete a fine-tuned model

    Args:
        model_id (str): Model ID to delete

    Returns:
        bool: True if successful, False otherwise
    """
    print("=" * 70)
    print(f"  DELETE MODEL: {model_id}")
    print("=" * 70)

    try:
        # Confirm deletion
        print(f"\n⚠️  WARNING: This will permanently delete the model!")
        print(f"   Model ID: {model_id}")
        print(f"\n   This action CANNOT be undone!")
        print(f"   You will lose access to this fine-tuned model.")

        confirm = input("\nType 'DELETE' to confirm: ").strip()

        if confirm != 'DELETE':
            print("❌ Deletion aborted")
            return False

        # Delete the model
        print("\n🗑️  Deleting model...")
        result = client.models.delete(model_id=model_id)

        print(f"✅ Model deleted successfully!")
        print(f"   {result}")

        return True

    except Exception as e:
        print(f"❌ Error deleting model: {e}")
        return False

# ============================================================
# Quick Action Functions (without confirmation)
# ============================================================

def quick_cancel_job(job_id):
    """
    Cancel a job without confirmation (use carefully!)

    Args:
        job_id (str): Job ID to cancel
    """
    try:
        cancelled_job = client.fine_tuning.jobs.cancel(job_id=job_id)
        print(f"✅ Job {job_id} cancelled. Status: {cancelled_job.status}")
        return True
    except Exception as e:
        print(f"❌ Error: {e}")
        return False

def quick_delete_model(model_id):
    """
    Delete a model without confirmation (use carefully!)

    Args:
        model_id (str): Model ID to delete
    """
    try:
        result = client.models.delete(model_id=model_id)
        print(f"✅ Model {model_id} deleted: {result}")
        return True
    except Exception as e:
        print(f"❌ Error: {e}")
        return False

# ============================================================
# Usage Examples
# ============================================================

if __name__ == "__main__":
    print("\n" + "=" * 70)
    print("  MISTRAL JOB & MODEL MANAGER")
    print("=" * 70)

    print("\n📚 Available Functions:\n")
    print("1. list_all_jobs()           - List all fine-tuning jobs")
    print("2. get_job_info(job_id)      - Get detailed info about a job")
    print("3. cancel_job(job_id)        - Cancel a running/queued job")
    print("4. list_all_models()         - List all fine-tuned models")
    print("5. delete_model(model_id)    - Delete a fine-tuned model")
    print("\n🚨 Quick actions (no confirmation):")
    print("6. quick_cancel_job(job_id)  - Cancel job immediately")
    print("7. quick_delete_model(id)    - Delete model immediately")

    print("\n" + "=" * 70)
    print("\n💡 Examples:")
    print("-" * 70)
    print("""
# List all jobs
jobs = list_all_jobs()

# Get info about a specific job
job = get_job_info("97603d41-284f-43a0-8801-cda54ac92e12")

# Cancel a job (with confirmation)
cancel_job("97603d41-284f-43a0-8801-cda54ac92e12")

# Or cancel immediately without confirmation
quick_cancel_job("97603d41-284f-43a0-8801-cda54ac92e12")

# List all fine-tuned models
models = list_all_models()

# Delete a model (with confirmation)
delete_model("ft:open-mistral-7b:xxx:20240430:xxx")

# Or delete immediately without confirmation
quick_delete_model("ft:open-mistral-7b:xxx:20240430:xxx")
    """)
    print("=" * 70)

    jobs = list_all_jobs()
    print(jobs)
    models = list_all_models()
    print(models)